# Example 2: LDA with NPR Articles
## CISB5123 Text Analytics - Lab 9: Topic Modeling


## Cell 1: Import Libraries

In [1]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel

# For data handling
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

print("Libraries imported successfully!")

Libraries imported successfully!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mohda\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mohda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mohda\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Cell 2: Load the Data

Load NPR articles from a CSV file

In [4]:
df = pd.read_csv('npr.csv')
documents = df['Article'].tolist()

print(f"Number of articles loaded: {len(documents)}")
print(f"\nFirst 3 articles:")
for i in range(3):
    print(f"{i}: {documents[i][:100]}...")

Number of articles loaded: 11992

First 3 articles:
0: In the Washington of 2016, even when the policy can be bipartisan, the politics cannot. And in that ...
1:   Donald Trump has used Twitter  —   his preferred means of communication  —   to weigh in on a swat...
2:   Donald Trump is unabashedly praising Russian President Vladimir Putin, a day after outgoing Presid...


## Cell 3: Preprocess the Data

This step:
- Tokenizes text into words
- Converts to lowercase
- Removes stopwords (common words)
- Lemmatizes words (converts to base form)

In [5]:
stop_words = set(stopwords.words('english'))  # Create a set of English stopwords
lemmatizer = WordNetLemmatizer()  # Initialize a WordNet lemmatizer

def preprocess_text(text):
    """
    Preprocess text by:
    1. Tokenizing into words and converting to lowercase
    2. Filtering out non-alphanumeric tokens
    3. Removing stopwords from the tokens
    4. Lemmatizing each token
    """
    tokens = word_tokenize(text.lower())  # Tokenize the text into words and convert to lowercase
    tokens = [token for token in tokens if token.isalnum()]  # Filter out non-alphanumeric tokens
    tokens = [token for token in tokens if token not in stop_words]  # Remove stopwords from the tokens
    tokens = [lemmatizer.lemmatize(token) for token in tokens]  # Lemmatize each token
    return tokens  # Return the preprocessed tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]  # Preprocess each document in the list

print("Data preprocessing complete!")
print(f"\nExample of preprocessed document (first article):")
print(preprocessed_documents[0])

Data preprocessing complete!

Example of preprocessed document (first article):
['washington', '2016', 'even', 'policy', 'bipartisan', 'politics', 'sense', 'year', 'show', 'little', 'sign', 'ending', 'president', 'obama', 'moved', 'sanction', 'russia', 'alleged', 'interference', 'election', 'concluded', 'republican', 'long', 'called', 'similar', 'severe', 'measure', 'could', 'scarcely', 'bring', 'approve', 'house', 'speaker', 'paul', 'ryan', 'called', 'obama', 'measure', 'appropriate', 'also', 'overdue', 'prime', 'example', 'administration', 'ineffective', 'foreign', 'policy', 'left', 'america', 'weaker', 'eye', 'gop', 'leader', 'sounded', 'much', 'theme', 'urging', 'president', 'obama', 'year', 'take', 'strong', 'action', 'deter', 'russia', 'worldwide', 'aggression', 'including', 'operation', 'wrote', 'devin', 'nunes', 'chairman', 'house', 'intelligence', 'committee', 'week', 'left', 'office', 'president', 'suddenly', 'decided', 'stronger', 'measure', 'indeed', 'appearing', 'cnn', 'fr

## Cell 4: Create document-term matrix

**Dictionary filtering:**
- `no_below=15`: Remove words appearing in fewer than 15 documents
- `no_above=0.5`: Remove words appearing in more than 50% of documents

In [6]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)

print(f"Dictionary created before filtering:")
print(f"Number of unique words: {len(dictionary)}")

# Filter out tokens that appear in less than 15 documents or more than 50% of the documents
dictionary.filter_extremes(no_below=15, no_above=0.5)

print(f"\nDictionary after filtering:")
print(f"Number of unique words: {len(dictionary)}")
print(f"(Removed words appearing in < 15 documents or > 50% of documents)")

# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

print(f"\nBag of Words created!")
print(f"Total documents in corpus: {len(corpus)}")
print(f"\nExample of first document as bag of words:")
print(f"Document 0: {corpus[0][:10]}...")  # Show first 10 word-count pairs

Dictionary created before filtering:
Number of unique words: 86155

Dictionary after filtering:
Number of unique words: 15974
(Removed words appearing in < 15 documents or > 50% of documents)

Bag of Words created!
Total documents in corpus: 11992

Example of first document as bag of words:
Document 0: [(0, 1), (1, 3), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1), (8, 4), (9, 1)]...


## Cell 5: Run LDA Model

**Parameters:**
- `num_topics=5`: Extract 5 topics
- `passes=15`: Iterate through the corpus 15 times

In [7]:
# Train an LDA model on the corpus with 5 topics using Gensim's LdaModel class
print("Training LDA model... this may take a moment...")
lda_model = LdaModel(corpus, num_topics=5, id2word=dictionary, passes=15)

print("LDA Model training complete!")
print(f"Model Parameters:")
print(f"  - Number of topics: 5")
print(f"  - Number of passes: 15")
print(f"  - Corpus size: {len(corpus)} documents")

Training LDA model... this may take a moment...
LDA Model training complete!
Model Parameters:
  - Number of topics: 5
  - Number of passes: 15
  - Corpus size: 11992 documents


## Cell 6: Interpret Results - Get Dominant Topic for Each Document

In [8]:
# empty list to store dominant topic labels for each document
article_labels = []

# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
    # for each document, convert to bag-of-words representation
    bow = dictionary.doc2bow(doc)
    # get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    # append to the list
    article_labels.append(dominant_topic)

# Create DataFrame
df_result = pd.DataFrame({"Article": documents, "Topic": article_labels})

# Print the DataFrame
print("Table with Articles and Topic:")
print(df_result)
print(f"\nTotal articles processed: {len(df_result)}")

Table with Articles and Topic:
                                                 Article  Topic
0      In the Washington of 2016, even when the polic...      0
1        Donald Trump has used Twitter  —   his prefe...      2
2        Donald Trump is unabashedly praising Russian...      0
3      Updated at 2:50 p. m. ET, Russian President Vl...      2
4      From photography, illustration and video, to d...      4
...                                                  ...    ...
11987  The number of law enforcement officers shot an...      2
11988    Trump is busy these days with victory tours,...      0
11989  It’s always interesting for the Goats and Soda...      1
11990  The election of Donald Trump was a surprise to...      0
11991  Voters in the English city of Sunderland did s...      3

[11992 rows x 2 columns]

Total articles processed: 11992


## Cell 7: Display Top Terms for Each Topic (Simple Format)

In [9]:
print("\n" + "="*80)
print("TOP TERMS FOR EACH TOPIC (Top 10 words per topic)")
print("="*80 + "\n")

for topic_id in range(lda_model.num_topics):
    print(f"Topic #{topic_id}:")
    top_terms = lda_model.show_topic(topic_id, topn=10)
    terms_list = [term[0] for term in top_terms]
    print(terms_list)
    print()


TOP TERMS FOR EACH TOPIC (Top 10 words per topic)

Topic #0:
['trump', 'clinton', 'president', 'republican', 'campaign', 'state', 'election', 'vote', 'obama', 'voter']

Topic #1:
['school', 'study', 'student', 'child', 'health', 'university', 'patient', 'food', 'water', 'disease']

Topic #2:
['police', 'report', 'state', 'government', 'court', 'country', 'law', 'case', 'official', 'told']

Topic #3:
['company', 'state', 'health', 'percent', 'million', 'care', 'plan', 'money', 'job', 'country']

Topic #4:
['know', 'think', 'thing', 'life', 'really', 'woman', 'story', 'world', 'day', 'back']



## Cell 8: Display Top Terms for Each Topic (With Weights)

In [10]:
print("="*80)
print("TOP TERMS FOR EACH TOPIC (With Weights)")
print("="*80 + "\n")

for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"  - {word.strip()} (weight: {weight.strip()})")
    print()

TOP TERMS FOR EACH TOPIC (With Weights)

Topic 0:
  - "trump" (weight: 0.034)
  - "clinton" (weight: 0.013)
  - "president" (weight: 0.011)
  - "republican" (weight: 0.009)
  - "campaign" (weight: 0.009)
  - "state" (weight: 0.008)
  - "election" (weight: 0.007)
  - "vote" (weight: 0.006)
  - "obama" (weight: 0.006)
  - "voter" (weight: 0.006)

Topic 1:
  - "school" (weight: 0.006)
  - "study" (weight: 0.006)
  - "student" (weight: 0.005)
  - "child" (weight: 0.005)
  - "health" (weight: 0.005)
  - "university" (weight: 0.004)
  - "patient" (weight: 0.004)
  - "food" (weight: 0.003)
  - "water" (weight: 0.003)
  - "disease" (weight: 0.003)

Topic 2:
  - "police" (weight: 0.006)
  - "report" (weight: 0.006)
  - "state" (weight: 0.005)
  - "government" (weight: 0.005)
  - "court" (weight: 0.004)
  - "country" (weight: 0.004)
  - "law" (weight: 0.004)
  - "case" (weight: 0.004)
  - "official" (weight: 0.004)
  - "told" (weight: 0.004)

Topic 3:
  - "company" (weight: 0.009)
  - "state" (w

## Cell 9: Topic Interpretation (Manual Analysis)

In [11]:
print("="*80)
print("TOPIC INTERPRETATION")
print("="*80 + "\n")

topics_interpretation = {
    0: "Education",
    1: "Personal",
    2: "World Politics",
    3: "Health",
    4: "US Election"
}

print("Proposed Topic Labels:")
for topic_id, label in topics_interpretation.items():
    print(f"Topic {topic_id}: {label}")

print("\nDistribution of articles across topics:")
topic_distribution = df_result['Topic'].value_counts().sort_index()
print(topic_distribution)

print("\n" + "="*80)
print("Summary:")
print("="*80)
print(f"Total articles analyzed: {len(df_result)}")
print(f"Topics discovered: 5")
print(f"Most common topic: Topic {topic_distribution.idxmax()} with {topic_distribution.max()} articles")
print(f"Least common topic: Topic {topic_distribution.idxmin()} with {topic_distribution.min()} articles")

TOPIC INTERPRETATION

Proposed Topic Labels:
Topic 0: Education
Topic 1: Personal
Topic 2: World Politics
Topic 3: Health
Topic 4: US Election

Distribution of articles across topics:
Topic
0    1764
1    2193
2    2614
3    1372
4    4049
Name: count, dtype: int64

Summary:
Total articles analyzed: 11992
Topics discovered: 5
Most common topic: Topic 4 with 4049 articles
Least common topic: Topic 3 with 1372 articles
